# Model 1 (Greedy Approach)

In [ ]:
import csv
from collections import defaultdict, deque, Counter

# ==========================================
# PART 1: OPTIMIZATION ALGORITHM (OPTIMIZER)
# ==========================================
def solve_weighted_greedy(input_data):
    """
    Generate a traffic light schedule based on traffic density.
    Input: Parsed data (streets, paths, I, D...)
    Output: Schedule dictionary {intersection_id: [(street_name, duration), ...]}
    """
    D, F, I, streets, paths = input_data
    
    # 1. Count traffic on each street
    street_counts = Counter()
    for path in paths:
        for street in path:
            street_counts[street] += 1
            
    # 2. Group streets by their ending intersection
    intersection_incoming = defaultdict(list)
    for name, info in streets.items():
        # info = (start, end, length)
        end_node = info[1]
        intersection_incoming[end_node].append(name)

    # 3. Build the schedule
    schedule = {}
    for i in range(I):
        incoming = intersection_incoming[i]
        
        # Remove streets that have no cars
        active_streets = [s for s in incoming if street_counts[s] > 0]
        
        if not active_streets:
            continue
            
        # Sort by descending traffic (most cars first)
        active_streets.sort(key=lambda s: street_counts[s], reverse=True)
        
        # Assign durations (Heuristic: more cars → more time, capped at 2s)
        min_cars = street_counts[active_streets[-1]]
        
        cycle = []
        for s in active_streets:
            cnt = street_counts[s]
            # If > 10× the lowest count → 2 seconds, else 1 second
            duration = 2 if cnt > min_cars * 10 else 1
            cycle.append((s, duration))
            
        schedule[i] = cycle
        
    return schedule

# ==========================================
# PART 2: SIMULATOR & SCORER
# ==========================================
def run_simulation(input_data, schedule):
    """
    Simulate vehicle movement second-by-second to calculate the score.
    """
    D, F, I, streets, paths = input_data
    
    # --- Data structures for simulation ---
    
    # 1. Car states
    # car_state[car_id] = (current_street_idx_in_path, arrival_time_at_street_end)
    # Initially, all cars start waiting at the end of the first street (idx = 0) at t=0.
    car_states = []
    for i in range(len(paths)):
        car_states.append({'idx': 0, 'arrive_time': 0})
        
    # 2. Queues for each street (FIFO)
    queues = defaultdict(deque)
    
    # Put each car into the queue of its starting street
    for car_id, path in enumerate(paths):
        start_street = path[0]
        queues[start_street].append(car_id)
        
    # 3. Pre-compute traffic light cycles
    inter_configs = {}
    for i, cycles in schedule.items():
        green_list = []
        for name, duration in cycles:
            green_list.extend([name] * duration)
        if green_list:
            inter_configs[i] = {
                'cycle_len': len(green_list),
                'sequence': green_list
            }

    # --- Time simulation loop (0 → D) ---
    total_score = 0
    cars_finished = 0
    
    # cars_waiting_to_move[t] = list of cars that finish a street and will join the next queue at time t
    future_arrivals = defaultdict(list)

    for t in range(D + 1):
        # A. Update cars arriving at the end of streets (scheduled earlier)
        if t in future_arrivals:
            for car_id, street_name in future_arrivals[t]:
                queues[street_name].append(car_id)
            del future_arrivals[t]  # Free memory

        # B. Process intersections (cars passing green lights)
        for i in range(I):
            if i not in inter_configs:
                continue
                
            config = inter_configs[i]
            
            # Determine which street has green light at time t
            t_mod = t % config['cycle_len']
            green_street = config['sequence'][t_mod]
            
            # Check if any car is waiting on this street
            if queues[green_street]:
                car_id = queues[green_street].popleft()
                
                # Car info
                car = car_states[car_id]
                path = paths[car_id]
                
                # Car has completed the current street (path[car['idx']])
                # Check if this is the last street
                if car['idx'] == len(path) - 1:
                    # Score = F + (D - t)
                    points = F + (D - t)
                    total_score += points
                    cars_finished += 1
                else:
                    # Move to the next street
                    next_street_name = path[car['idx'] + 1]
                    travel_time = streets[next_street_name][2]  # L
                    
                    # Update car state
                    car['idx'] += 1
                    arrival_time = t + travel_time
                    
                    if arrival_time <= D:
                        future_arrivals[arrival_time].append((car_id, next_street_name))
                        
    return total_score, cars_finished

# ==========================================
# MAIN DRIVER
# ==========================================
def main():
    input_file = 'hashcode.csv'
    output_file = 'submission_scored.txt'
    
    print("--- Starting process ---")
    
    with open(input_file, 'r') as f:
        reader = csv.reader(f)
        rows = list(reader)
        
    # Parse header
    D, I, S, V, F = map(int, rows[0])
    
    # Parse streets
    # Format: start, end, name, L
    streets = {}  # name -> (start, end, L)
    idx = 1
    for _ in range(S):
        row = rows[idx]
        streets[row[2]] = (int(row[0]), int(row[1]), int(row[3]))
        idx += 1
        
    # Parse paths
    # Format: P, name1, name2...
    paths = []
    for _ in range(V):
        paths.append(rows[idx][1:])
        idx += 1
        
    input_data = (D, F, I, streets, paths)
    print(f"Data loaded: {I} intersections, {S} streets, {V} cars.")

    # 2. Run optimizer
    print("Running optimization algorithm...")
    schedule = solve_weighted_greedy(input_data)
    
    # 3. Write output file (HashCode format)
    with open(output_file, 'w') as f:
        f.write(f"{len(schedule)}\n")
        for i, cycle in schedule.items():
            f.write(f"{i}\n")
            f.write(f"{len(cycle)}\n")
            for name, duration in cycle:
                f.write(f"{name} {duration}\n")
    print(f"Submission written to: {output_file}")

    # 4. Run simulation to compute score
    print("Running simulation for scoring...")
    score, finished_count = run_simulation(input_data, schedule)
    
    print("-" * 30)
    print("SIMULATION RESULTS:")
    print(f"Cars finished: {finished_count} / {V}")
    print(f"TOTAL SCORE: {score}")
    print("-" * 30)

if __name__ == "__main__":
    main()


# Model 2 (Mixed-Integer Programming)

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import collections
import math

# --- Data Structures and Input Parsing ---

def parse_input(filename):
    """
    Parses the traffic simulation input file.
    """
    data = {}
    with open(filename, 'r') as f:
        lines = f.readlines()

    # 1. First line: D, I, S, V, F
    D, I, S, V, F = map(int, lines[0].split())
    data['D'] = D  # Simulation duration
    data['I'] = I  # Number of intersections (0 to I-1)
    data['S'] = S  # Number of streets
    data['V'] = V  # Number of cars
    data['F'] = F  # Bonus points

    # 2. Street descriptions (S lines)
    streets_data = {}
    lines_idx = 1
    for i in range(S):
        parts = lines[lines_idx].split()
        B = int(parts[0])  # Start intersection
        E = int(parts[1])  # End intersection
        name = parts[2]     # Street name
        L = int(parts[3])  # Length/Time

        streets_data[name] = {
            'B': B, 'E': E, 'L': L, 'name': name
        }
        lines_idx += 1
    data['streets'] = streets_data

    # 3. Car paths (V lines)
    cars_data = []
    for i in range(V):
        parts = lines[lines_idx].split()
        P = int(parts[0])  # Path length
        path = parts[1:P+1]
        cars_data.append({
            'P': P, 'path': path
        })
        lines_idx += 1
    data['cars'] = cars_data

    return data

def calculate_demand(data):
    """
    Calculates the total number of cars that use each street, representing the demand.
    This demand drives the Gurobi optimization model.
    """
    street_demand = collections.defaultdict(int)

    # Iterate over all cars
    for car in data['cars']:
        # Cars use all streets in their path *except* the last one (they don't queue there)
        path_for_queuing = car['path'][:-1]
        for street_name in path_for_queuing:
            street_demand[street_name] += 1

    return street_demand

# --- Gurobi Optimization Model (Heuristic Assignment) ---

def optimize_schedule_durations(data, street_demand):
    """
    Uses Gurobi to assign a minimum green light duration (t_s) to streets
    that have high car demand (D_s).

    This is a simplification (a linear program) because the true dynamic
    queuing and cycle time relationship is non-linear and non-convex.

    Objective: Minimize the total green time assigned.
    Constraint: Green time must be at least 1, and for high-demand streets,
                it must be proportional to the demand.
    """

    # 1. Pre-process street data for optimization
    streets = list(data['streets'].keys())

    # Maximum demand value for normalization
    max_demand = max(street_demand.values()) if street_demand else 1
    # Big M value (max duration is D, but let's use a safe upper bound)
    D_max = data['D']

    print("Building Gurobi model...")
    m = gp.Model("TrafficScheduleHeuristic")
    m.setParam('OutputFlag', 0) # Suppress Gurobi output

    # 2. Variables
    # t_s: Integer, green light duration for street s.
    t = m.addVars(streets, vtype=GRB.INTEGER, name="GreenTime", lb=0)
    # x_s: Binary, 1 if street s is included in the schedule, 0 otherwise.
    x = m.addVars(streets, vtype=GRB.BINARY, name="Scheduled")

    # 3. Objective: Minimize total green time used (proxy for minimizing cycle length)
    m.setObjective(t.sum(), GRB.MINIMIZE)

    # 4. Constraints
    for s in streets:
        demand = street_demand[s]

        # Constraint 1: Big M linkage - If t_s > 0, then x_s must be 1.
        m.addConstr(t[s] <= D_max * x[s], f"BigM_Upper_{s}")

        # Constraint 2: If street is scheduled (x_s=1), min green time is 1.
        m.addConstr(t[s] >= x[s], f"MinTime_Lower_{s}")

        # Constraint 3: Demand-based Heuristic Assignment (Tune this factor!)
        # Streets with demand must be scheduled (x_s=1)
        if demand > 0:
            m.addConstr(x[s] == 1, f"MustSchedule_{s}")

            # Heuristic: Green time is proportional to demand, ensuring at least 1.
            # Factor of 10 is an arbitrary tuning parameter: D_s/10
            # Higher factor = shorter green lights, tighter cycles, but higher queue risk.
            required_time = max(1, math.ceil(demand / 10))
            m.addConstr(t[s] >= required_time, f"DemandProp_{s}")

    # 5. Solve the model
    m.optimize()

    # 6. Extract the solution
    schedule_durations = {}
    if m.status == GRB.OPTIMAL:
        for s in streets:
            duration = int(t[s].X)
            if duration > 0:
                schedule_durations[s] = duration

    return schedule_durations

# --- Solution Formatting ---

def create_submission_file(data, schedule_durations):
    """
    Formats the Gurobi solution into the required submission file format.
    The order of streets in the schedule is a secondary heuristic (e.g., alternating
    based on a simple rule, but here we'll just use the order of appearance).
    """

    # 1. Group streets by the intersection they lead into
    intersection_schedules = collections.defaultdict(list)
    for s_name, duration in schedule_durations.items():
        street = data['streets'][s_name]
        # Street leads *into* intersection E
        intersection_id = street['E']

        # Only include streets with a positive duration assigned by Gurobi
        if duration > 0:
            intersection_schedules[intersection_id].append((s_name, duration))

    # 2. Build the output content
    output_lines = []

    # A: Number of intersections with a schedule
    output_lines.append(f"{len(intersection_schedules)}")

    for i_id, schedules in intersection_schedules.items():
        # i: Intersection ID
        output_lines.append(f"{i_id}")

        # E_i: Number of incoming streets covered
        output_lines.append(f"{len(schedules)}")

        # E_i lines: street name and duration T
        for s_name, duration in schedules:
            output_lines.append(f"{s_name} {duration}")

    return "\n".join(output_lines)


# --- Main Execution ---

def main(input_filename="hashcode.in.txt", output_filename="submission.txt"):
    try:
        # 1. Parse Input
        print(f"Reading input from {input_filename}...")
        data = parse_input(input_filename)

        # 2. Calculate Demand (Input for Gurobi)
        street_demand = calculate_demand(data)

        # 3. Optimize Schedule Durations using Gurobi
        schedule_durations = optimize_schedule_durations(data, street_demand)

        # 4. Format Output
        submission_content = create_submission_file(data, schedule_durations)

        # 5. Write to File
        with open(output_filename, 'w') as f:
            f.write(submission_content)

        print(f"Optimization complete. Schedule written to {output_filename}.")
        print("\n--- Generated Submission File Content ---\n")
        # Print only the first 20 lines to avoid overwhelming the output for large files
        print('\n'.join(submission_content.splitlines()[:20]))
        if len(submission_content.splitlines()) > 20:
             print("[... Submission truncated for brevity ...]")
        print("\n-----------------------------------------\n")

    except FileNotFoundError:
        print(f"Error: Input file '{input_filename}' not found. Please ensure it exists.")

if __name__ == '__main__':
    # You must have Gurobi installed and licensed to run this.
    main()

In [ ]:
import collections
import math
import sys
import gurobipy as gp
from gurobipy import GRB

# Increase recursion limit for deep dictionary/list copies in large datasets
sys.setrecursionlimit(2000)

# --- GLOBAL DATA STRUCTURES ---
# D: Simulation Duration, F: Bonus
D, F = 0, 0
STREETS = {}
CARS_DATA = []
INTERSECTIONS = {}

# --- HELPER CLASSES ---

class Car:
    """Represents a single car's state in the simulation."""
    def __init__(self, car_id, path):
        self.id = car_id
        self.path = path  # List of street names to follow
        self.path_index = 0
        self.current_street_name = path[0]
        self.current_street_length = STREETS[path[0]]['L']

        # Position on the current street: 0 is at the start (just entered the street),
        # Length L is at the end (waiting at the light).
        self.position = 0

        # State: 'MOVING', 'WAITING', 'FINISHED', 'IDLE' (before starting)
        self.status = 'IDLE'
        self.start_time = 0
        self.finish_time = -1

    def get_destination_intersection(self):
        """Returns the ID of the intersection at the end of the current street."""
        if self.path_index < len(self.path):
            return STREETS[self.current_street_name]['E']
        return -1 # Finished

    def move(self):
        """Advances the car one unit (one second)."""
        if self.status == 'MOVING':
            self.position += 1

    def next_street(self, current_time):
        """Moves the car to the next street in its path."""
        self.path_index += 1

        # Check if the path is complete
        if self.path_index >= len(self.path):
            self.status = 'FINISHED'
            self.finish_time = current_time
            return

        # Move to the new street
        self.current_street_name = self.path[self.path_index]
        self.current_street_length = STREETS[self.current_street_name]['L']
        self.position = 1 # Car immediately moves 1 unit into the new street
        self.status = 'MOVING'

    def wait(self):
        """Sets the car's status to WAITING (at the end of a street)."""
        self.status = 'WAITING'

class Intersection:
    """Represents a single intersection with its traffic light schedule."""
    def __init__(self, i_id):
        self.id = i_id
        # {street_name: [T, start_time, end_time]}
        self.schedule = []
        self.queues = collections.defaultdict(collections.deque) # {street_name: deque([Car_1, Car_2, ...])}
        self.cycle_time = 0
        self.current_green_street = None
        self.time_in_cycle = 0 # Current time modulo cycle_time

    def set_schedule(self, schedule_list):
        """Loads the schedule from the submission file."""
        self.schedule = schedule_list
        self.cycle_time = sum(duration for _, duration in schedule_list)

    def update_light(self, current_time):
        """Determines which street is green at the current time T."""
        if self.cycle_time == 0 or not self.schedule:
            self.current_green_street = None
            return

        # Calculate time within the cycle
        self.time_in_cycle = current_time % self.cycle_time

        # Find the currently green street
        cumulative_time = 0
        for street_name, duration in self.schedule:
            if self.time_in_cycle >= cumulative_time and self.time_in_cycle < cumulative_time + duration:
                self.current_green_street = street_name
                return
            cumulative_time += duration

        # Should not happen if logic is correct, but default to None
        self.current_green_street = None

# --- INITIALIZATION AND PARSING ---

def parse_input(filename):
    """Parses the traffic simulation input file and initializes global data."""
    global D, F, STREETS, CARS_DATA, INTERSECTIONS

    with open(filename, 'r') as f:
        lines = f.readlines()

    # 1. First line: D, I, S, V, F
    D, I, S, V, F = map(int, lines[0].split())

    # Initialize all intersections
    for i in range(I):
        INTERSECTIONS[i] = Intersection(i)

    # 2. Street descriptions (S lines)
    lines_idx = 1
    for i in range(S):
        B, E, name, L = lines[lines_idx].split()
        B, E, L = int(B), int(E), int(L)
        STREETS[name] = {'B': B, 'E': E, 'L': L, 'name': name}
        lines_idx += 1

    # 3. Car paths (V lines)
    for i in range(V):
        parts = lines[lines_idx].split()
        path = parts[1:]
        CARS_DATA.append(Car(i, path))
        lines_idx += 1

    # Set the initial status for cars that can start moving
    for car in CARS_DATA:
        if car.current_street_length == 0:
            # Handle the highly unlikely case of 0-length streets (instant travel)
            # This car effectively starts at the end of its first street.
            car.position = 0 # It will move to the next street immediately
        else:
            car.status = 'MOVING' # All cars start moving on their first street

    print(f"Data Loaded. D={D}, F={F}, Streets={len(STREETS)}, Cars={len(CARS_DATA)}, Intersections={len(INTERSECTIONS)}")

def parse_submission(filename):
    """Loads the schedule from the submission file."""
    try:
        with open(filename, 'r') as f:
            lines = f.read().splitlines()
    except FileNotFoundError:
        print(f"Error: Submission file '{filename}' not found.")
        return

    if not lines: return

    A = int(lines[0]) # Number of scheduled intersections
    line_idx = 1

    for _ in range(A):
        i_id = int(lines[line_idx])
        line_idx += 1
        E_i = int(lines[line_idx])
        line_idx += 1

        schedule = []
        for _ in range(E_i):
            street_name, duration = lines[line_idx].split()
            schedule.append((street_name, int(duration)))
            line_idx += 1

        if i_id in INTERSECTIONS:
            INTERSECTIONS[i_id].set_schedule(schedule)

    print(f"Schedule Loaded. {A} intersections have defined cycles.")

# --- SIMULATION LOGIC ---

def run_simulation():
    """Runs the second-by-second simulation and calculates the score."""
    global D

    # Initialize total score
    total_score = 0

    # 1. Initial Car Setup (Car must move its initial street first)
    # Cars starting on a street of length L, take L seconds to reach the first light.

    for t in range(1, D + 1):

        # --- A. Update Traffic Lights ---
        # Update the green street for all intersections based on the current time t
        for i_id, intersection in INTERSECTIONS.items():
            intersection.update_light(t)

        # --- B. Process Cars at Lights (Green Light Action) ---
        # Determine which cars get to pass the light this second.

        cars_to_move_from_queue = [] # List to track cars that pass the light

        for i_id, intersection in INTERSECTIONS.items():
            green_street = intersection.current_green_street

            if green_street and intersection.queues[green_street]:
                # If the green street has waiting cars, the first car passes
                car = intersection.queues[green_street].popleft()
                cars_to_move_from_queue.append(car)

        # Now, move the cars that passed the light
        for car in cars_to_move_from_queue:
             car.next_street(t) # Moves car into the next street (pos=1, status=MOVING)


        # --- C. Move All Moving Cars ---
        # Advance every MOVING car one step (1 second)

        cars_to_queue_next_step = [] # Cars that reach the end of their street

        for car in CARS_DATA:
            if car.status == 'MOVING':
                car.move()

                # Check if the car reached the end of its current street
                if car.position >= car.current_street_length:

                    if car.path_index == len(car.path) - 1:
                        # Reached the end of the final street
                        car.status = 'FINISHED'
                        car.finish_time = t
                        total_score += F + (D - car.finish_time)
                    else:
                        # Reached the end of a non-final street, now waiting at the light
                        car.position = car.current_street_length # Normalize position to the end
                        car.wait()
                        cars_to_queue_next_step.append(car)

        # --- D. Enqueue Waiting Cars ---
        # Cars that just arrived at a light need to be placed in the queue
        for car in cars_to_queue_next_step:
            dest_i = car.get_destination_intersection()
            if dest_i != -1: # Ensure it's a valid intersection
                street_name = car.current_street_name
                INTERSECTIONS[dest_i].queues[street_name].append(car)


        # --- E. Progress Report ---
        if t % 500 == 0:
            finished_count = sum(1 for car in CARS_DATA if car.status == 'FINISHED')
            print(f"Time T={t}/{D}. Finished Cars: {finished_count}. Current Score: {total_score}")

    # --- F. Final Score Tally (Final check for score) ---
    # The score formula includes the check T_finish <= D, which is handled implicitly by the loop structure.

    return total_score

# --- Main Execution ---

def main():
    global D, F
    input_filename = "hashcode.in.txt"
    submission_filename = "submission.txt"

    # 1. Load Data
    try:
        parse_input(input_filename)
        parse_submission(submission_filename)
    except FileNotFoundError as e:
        print(f"CRITICAL ERROR: {e}. Ensure both '{input_filename}' and '{submission_filename}' are available.")
        return

    # 2. Run Simulation
    print("\n--- Starting Simulation ---")
    final_score = run_simulation()

    # 3. Print Results
    print("\n==============================================")
    print(f"SIMULATION COMPLETE at T={D}")
    print(f"FINAL TOTAL SCORE: {final_score}")
    print(f"Total Cars: {len(CARS_DATA)}")
    print(f"Cars Finished: {sum(1 for car in CARS_DATA if car.status == 'FINISHED')}")
    print("==============================================")


if __name__ == '__main__':
    main()

# Model 3 (Demand-Weighted Local Search)

In [ ]:
import sys
from collections import defaultdict
from bisect import bisect_right

# ============================================================
# Data structures
# ============================================================

class Street:
    def __init__(self, unique_id, start, end, name, length):
        self.unique_id = unique_id
        self.start = start
        self.end = end
        self.name = name
        self.length = length
        self.incoming_usage_count = 0
        self.cars_on_start = 0


class Car:
    def __init__(self, unique_id, streets):
        self.unique_id = unique_id
        self.streets = streets  # list[Street]

    def time_need_to_drive(self):
        # sum of lengths from street index 1 onwards
        # (i.e., total travel after reaching first intersection)
        t = 0
        for i in range(1, len(self.streets)):
            t += self.streets[i].length
        return t


class Intersection:
    def __init__(self, idx):
        self.id = idx
        self.incoming = []  # list[Street]
        self.outgoing = []  # list[Street]


class CarSimulationPosition:
    """
    Stores precomputed arrays of:
      - street ids
      - street end intersections
      - street lengths
    """
    def __init__(self, car, time_got_here):
        self.car = car
        self.street_number = 0
        self.time_got_here = time_got_here
        self.time_left_on_drive = car.time_need_to_drive()

        self.street_ids = []
        self.street_ends = []
        self.street_lengths = []

        for st in self.car.streets:
            self.street_ids.append(st.unique_id)
            self.street_ends.append(st.end)
            self.street_lengths.append(st.length)

    def init(self, time_got_here):
        """Reset for a new simulation run."""
        self.street_number = 0
        self.time_got_here = time_got_here
        self.time_left_on_drive = self.car.time_need_to_drive()


class Problem:
    def __init__(self):
        self.duration = 0
        self.bonus_per_car = 0
        self.intersections = []  # list[Intersection]
        self.streets = []        # list[Street]
        self.cars = []           # list[Car]

    @staticmethod
    def load_problem(file_name):
        """
        Hash Code 2021 input format.
        """
        with open(file_name, "r") as f:
            first = f.readline().split()
            d = int(first[0])
            number_of_intersections = int(first[1])
            s = int(first[2])
            v = int(first[3])
            f_bonus = int(first[4])

            problem = Problem()
            problem.duration = d
            problem.bonus_per_car = f_bonus

            # Read streets
            streets_map = {}
            for street_id in range(s):
                line = f.readline().split()
                start = int(line[0])
                end = int(line[1])
                name = line[2]
                length = int(line[3])
                st = Street(street_id, start, end, name, length)
                problem.streets.append(st)
                streets_map[name] = st

            # Read cars
            for car_id in range(v):
                parts = f.readline().split()
                p = int(parts[0])
                street_names = parts[1:]
                car_streets = [streets_map[name] for name in street_names]
                problem.cars.append(Car(car_id, car_streets))

        # Build intersections
        problem.intersections = [Intersection(i) for i in range(number_of_intersections)]
        for st in problem.streets:
            problem.intersections[st.start].outgoing.append(st)
            problem.intersections[st.end].incoming.append(st)

        # Count incoming usage (for completeness)
        for car in problem.cars:
            for i in range(len(car.streets) - 1):
                car.streets[i].incoming_usage_count += 1

        # Cars on start
        for st in problem.streets:
            st.cars_on_start = 0
        for car in problem.cars:
            car.streets[0].cars_on_start += 1

        return problem

    def remove_unused_streets(self):
        """
        remove incoming streets from intersection
        if their IncomingUsageCount == 0.
        """
        removed = 0
        for inter in self.intersections:
            keep = []
            for st in inter.incoming:
                if st.incoming_usage_count > 0:
                    keep.append(st)
                else:
                    removed += 1
            inter.incoming = keep
        return removed


class GreenLightCycle:
    def __init__(self, street, duration):
        self.street = street        # Street object
        self.duration = duration    # int
        # 'GreenLightUsed' is only used in the fancy optimizeGreenLightOrder4,
        # not in the lite simulation, so we skip it here.


class SolutionIntersection:
    def __init__(self, idx):
        self.id = idx
        self.green_cycles = []      # list[GreenLightCycle]
        self.green_array = []       # list[street_id] (expanded cycle)
        self.last_car_pass_time = -1

    def build_green_array(self):
        """
        Flatten cycles into an array where index t % len(green_array)
        tells which street_id is green at that time.
        """
        self.green_array = []
        for cycle in self.green_cycles:
            for _ in range(cycle.duration):
                self.green_array.append(cycle.street.unique_id)


class Solution:
    def __init__(self, num_intersections):
        self.intersections = [SolutionIntersection(i) for i in range(num_intersections)]

    def count_intersections_with_green_lights(self):
        c = 0
        for si in self.intersections:
            if any(cyc.duration > 0 for cyc in si.green_cycles):
                c += 1
        return c


# ============================================================
# Basic initial solution: 1 second per incoming street
# ============================================================

def init_basic_solution(problem, solution):
    """
    - For each intersection i:
      - one GreenLightCycle per incoming street, duration=1
    """
    for inter in problem.intersections:
        si = solution.intersections[inter.id]
        si.green_cycles.clear()
        for st in inter.incoming:
            si.green_cycles.append(GreenLightCycle(street=st, duration=1))
import math
from collections import defaultdict

def build_weighted_solution(problem):
    """
    Build a better-than-naive initial schedule:
    - Give more green seconds to streets with more "finishable" cars.
    - Keep one cycle per intersection.
    """
    D = problem.duration
    bonus = problem.bonus_per_car

    solution = Solution(num_intersections=len(problem.intersections))

    # 1) Find finishable cars and count usage per street
    street_weight = defaultdict(float)

    for car in problem.cars:
        total_time = car.time_need_to_drive()
        if total_time > D:
            continue  # this car can never finish, ignore

        # weight of this car could be 1, or (D - total_time) to favor quick paths
        w = 1.0
        # or try: w = 1.0 + (D - total_time) / D

        for i in range(len(car.streets) - 1):  # only internal streets
            st = car.streets[i]
            street_weight[st.unique_id] += w

    # 2) For each intersection, distribute green time among its incoming streets
    for inter in problem.intersections:
        si = solution.intersections[inter.id]
        incoming = inter.incoming

        if not incoming:
            continue

        # Get weights for incoming streets, fallback to 0
        weights = [street_weight[st.unique_id] for st in incoming]
        total_w = sum(weights)

        if total_w == 0:
            # No meaningful weight info -> fallback to uniform 1s
            for st in incoming:
                si.green_cycles.append(GreenLightCycle(street=st, duration=1))
            continue

        # Choose total cycle length (small-ish to react quickly)
        # try between 4 and 10
        TOTAL_CYCLE = min(10, max(3, len(incoming)))

        # duration_i = max(1, round(TOTAL_CYCLE * w_i / total_w))
        durations = []
        for w in weights:
            dur = max(1, int(round(TOTAL_CYCLE * w / total_w)))
            durations.append(dur)

        # If rounding overshot the total, trim a bit
        extra = sum(durations) - TOTAL_CYCLE
        while extra > 0 and any(d > 1 for d in durations):
            for idx, d in enumerate(durations):
                if extra == 0:
                    break
                if d > 1:
                    durations[idx] -= 1
                    extra -= 1

        # Add cycles (we can also sort by weight desc)
        order = sorted(range(len(incoming)), key=lambda idx: -weights[idx])
        for idx in order:
            st = incoming[idx]
            dur = durations[idx]
            si.green_cycles.append(GreenLightCycle(street=st, duration=dur))

    return solution

import random
import copy

def clone_solution(solution):
    new_sol = Solution(len(solution.intersections))
    for i, si in enumerate(solution.intersections):
        new_si = new_sol.intersections[i]
        for cyc in si.green_cycles:
            new_si.green_cycles.append(
                GreenLightCycle(cyc.street, cyc.duration)
            )
    return new_sol

def local_search(problem, initial_solution, iters=1000, print_every=100):
    """
    Simple hill-climbing:
    - Start from initial_solution
    - Randomly tweak intersection schedules
    - Keep modifications that improve score
    """
    best_solution = clone_solution(initial_solution)
    best_score = run_simulation_lite(problem, best_solution)
    print(f"Initial score: {best_score}")

    for it in range(1, iters + 1):
        # Work on a copy
        candidate = clone_solution(best_solution)

        # Pick a random intersection that has at least 1 green cycle
        idxs = [i for i, si in enumerate(candidate.intersections) if len(si.green_cycles) > 0]
        if not idxs:
            break
        i = random.choice(idxs)
        si = candidate.intersections[i]

        # Decide move type: 0 = swap order, 1 = change duration
        move_type = random.randint(0, 1)

        if move_type == 0 and len(si.green_cycles) >= 2:
            # Swap two random phases
            a, b = random.sample(range(len(si.green_cycles)), 2)
            si.green_cycles[a], si.green_cycles[b] = si.green_cycles[b], si.green_cycles[a]

        else:
            # Change duration of a random phase
            j = random.randrange(len(si.green_cycles))
            old_dur = si.green_cycles[j].duration
            if old_dur == 1:
                # either stay or increase
                si.green_cycles[j].duration = old_dur + 1
            else:
                # +/- 1 with 50-50
                if random.random() < 0.5:
                    si.green_cycles[j].duration = max(1, old_dur - 1)
                else:
                    si.green_cycles[j].duration = old_dur + 1

        # Evaluate candidate
        candidate_score = run_simulation_lite(problem, candidate)

        if candidate_score > best_score:
            best_solution = candidate
            best_score = candidate_score
            if print_every is not None:
                print(f"[iter {it}] Improved: {best_score}")

        if print_every and it % print_every == 0:
            print(f"[iter {it}] current best: {best_score}")

    return best_solution, best_score


# ============================================================
# Simulation (lite version)
# ============================================================

def run_simulation_lite_with_stats(problem, solution):
    """
    Discrete-time simulation from t=0..Problem.Duration.

    Returns:
        {
            "score": total_score,
            "finished_cars": number_of_cars_that_finished_in_time,
            "total_time_saved": sum_over_finished_cars_of(D - finish_time)
        }
    """
    D = problem.duration
    bonus = problem.bonus_per_car
    cars = problem.cars

    # Build expanded green arrays per intersection
    for si in solution.intersections:
        si.build_green_array()
        si.last_car_pass_time = -1

    # Prepare CarSimulationPosition instances, one per car
    car_positions = []
    n = len(cars)
    simulation_car_start = -(n + 1)  # start times negative so they are available at t=0
    for car in cars:
        cp = CarSimulationPosition(car, simulation_car_start)
        car_positions.append(cp)
        simulation_car_start += 1

    # Sort by TimeGotHere ascending 
    car_positions.sort(key=lambda c: c.time_got_here)

    total_score = 0
    finished_cars = 0
    total_time_saved = 0
    current_time = 0

    while current_time <= D:
        i = 0
        # Iterate cars in order of arrival time
        while i < n:
            cp = car_positions[i]
            # If this car arrives in the future, all later cars do too
            if cp.time_got_here > current_time:
                break

            # Determine intersection where car is waiting:
            # It is at end of its "current" street index street_number
            inter_id = cp.street_ends[cp.street_number]
            si = solution.intersections[inter_id]

            # Check if this intersection already used this time step
            if si.last_car_pass_time == current_time:
                i += 1
                continue

            if len(si.green_array) == 0:
                i += 1
                continue

            # Which street id is green now?
            green_street_id = si.green_array[current_time % len(si.green_array)]

            # Is this car's incoming street the green one?
            if cp.street_ids[cp.street_number] != green_street_id:
                i += 1
                continue

            # Intersection is now used at this time step
            si.last_car_pass_time = current_time

            # Car passes intersection and enters the next street
            cp.street_number += 1
            new_len = cp.street_lengths[cp.street_number]
            cp.time_got_here = current_time + new_len
            cp.time_left_on_drive -= new_len

            # Check if car finished its route
            if cp.street_number == len(cp.car.streets) - 1:
                # If finished on time, award score and update stats
                if cp.time_got_here <= D:
                    time_saved = D - cp.time_got_here
                    total_score += bonus + time_saved
                    finished_cars += 1
                    total_time_saved += time_saved

                # Move finished car to the end (so it is never processed again)
                cp.time_got_here = D + 1  # put far in future
                car_positions.pop(i)
                car_positions.append(cp)
                # n unchanged; next car is now at index i, so do not increment
                continue
            else:
                # Car not finished; we must keep car_positions sorted by time_got_here.
                car_positions.pop(i)
                # Find new insertion index
                new_pos = bisect_right(
                    [c.time_got_here for c in car_positions],
                    cp.time_got_here
                )
                car_positions.insert(new_pos, cp)
                # Next car is now at index i, so do not increment
                continue

            # Default: go to next car
            i += 1

        current_time += 1

    return {
        "score": total_score,
        "finished_cars": finished_cars,
        "total_time_saved": total_time_saved,
    }


def run_simulation_lite(problem, solution):
    """
    Backwards-compatible wrapper used by local_search etc.
    Still returns only the score.
    """
    stats = run_simulation_lite_with_stats(problem, solution)
    return stats["score"]

# ============================================================
# Example usage
# ============================================================

if __name__ == "__main__":
    INPUT_FILE = "hashcode.txt"
    problem = Problem.load_problem(INPUT_FILE)
    print("Duration:", problem.duration)
    print("Intersections:", len(problem.intersections))
    print("Streets:", len(problem.streets))
    print("Cars:", len(problem.cars))

    removed = problem.remove_unused_streets()
    print("Streets removed (unused as incoming):", removed)

    solution0 = build_weighted_solution(problem)
    best_sol, best_score = local_search(problem, solution0, iters=1000)
    print("Final best score (search):", best_score)

    # Re-simulate best solution to get detailed stats
    stats = run_simulation_lite_with_stats(problem, best_sol)

    print("Final best score (recomputed):", stats["score"])
    print(
        f"Cars finished within deadline {problem.duration} s: "
        f"{stats['finished_cars']} out of {len(problem.cars)}"
    )
    print("Total time saved across all finished cars:", stats["total_time_saved"])
    if stats["finished_cars"] > 0:
        print(
            "Average time saved per finished car:",
            stats["total_time_saved"] / stats["finished_cars"]
        )


# Model 4 (Time ordered Scheduling)

In [ ]:
import sys
from collections import defaultdict, Counter, deque

# --- CONFIGURATION ---
INPUT_FILENAME = 'hashcode.in.txt'
OUTPUT_FILENAME = 'submission_optimized.out'
PRINT_INTERVAL = 500 

def solve_time_ordered_greedy():
    # ==========================================
    # 1. PARSE INPUT
    # ==========================================
    print(f"Reading {INPUT_FILENAME}...")
    try:
        with open(INPUT_FILENAME, 'r') as f:
            lines = f.read().splitlines()
    except FileNotFoundError:
        print("File not found.")
        return

    iterator = iter(lines)
    D, I, S, V, F = map(int, next(iterator).split())
    
    street_map = {}  # name -> (end_node, length)
    intersection_in = defaultdict(list)
    
    for _ in range(S):
        parts = next(iterator).split()
        name = parts[2]
        E = int(parts[1])
        L = int(parts[3])
        street_map[name] = (E, L)
        intersection_in[E].append(name)
        
    car_paths = []
    for _ in range(V):
        parts = next(iterator).split()
        car_paths.append(parts[1:])

    # ==========================================
    # 2. ADVANCED ANALYSIS
    # ==========================================
    print("Analyzing traffic flow & timing...")
    
    street_importance = Counter()
    street_arrival_times = defaultdict(list)
    
    # We simulate "free flow" travel to guess when cars arrive at each street
    # This helps us order the green lights correctly (First-Arrive, First-Serve)
    
    for path in car_paths:
        # 1. Filter Impossible Paths
        min_travel = sum(street_map[s][1] for s in path)
        if min_travel > D: continue
            
        # 2. Calculate Weight (High Value for Short Paths)
        weight = 1 + (10000 / (min_travel + 1))
        
        current_time = 0
        for street in path:
            street_importance[street] += weight
            
            # Record expected arrival time at the END of this street
            travel_time = street_map[street][1]
            current_time += travel_time
            street_arrival_times[street].append(current_time)

    # ==========================================
    # 3. BUILD OPTIMIZED SCHEDULE
    # ==========================================
    print("Generating Time-Ordered Schedule...")
    
    intersection_schedules = {}
    
    for i_id in range(I):
        incoming = intersection_in[i_id]
        active = [s for s in incoming if street_importance[s] > 0]
        
        if not active:
            continue
            
        # --- STEP A: Determine Order ---
        # Sort streets by WHEN cars arrive. 
        # If Street A has cars arriving at T=10, and Street B at T=50, A goes first.
        # We use the 10th percentile arrival time to capture the "leading edge" of the platoon.
        def get_earliness(street_name):
            times = street_arrival_times[street_name]
            if not times: return float('inf')
            times.sort()
            # Take the arrival time of the first few cars
            idx = min(len(times)-1, 0) 
            return times[idx]
            
        active.sort(key=get_earliness)
        
        # --- STEP B: Determine Duration ---
        total_imp = sum(street_importance[s] for s in active)
        cycle = []
        
        for street in active:
            imp = street_importance[street]
            ratio = imp / total_imp
            
            if ratio > 0.5:
                duration = 2
            else:
                duration = 1
                
            cycle.append((street, duration))
            
        intersection_schedules[i_id] = cycle

    # ==========================================
    # 4. SIMULATION (VALIDATION)
    # ==========================================
    print("\nRunning Simulation...")
    print("-" * 50)
    print(f"{'Time':<10} | {'Cars Finished':<15} | {'Score':<10}")
    print("-" * 50)
    
    # -- Sim State --
    queues = defaultdict(deque)
    car_props = []
    
    # Init cars
    for c_idx, path in enumerate(car_paths):
        start_street = path[0]
        queues[start_street].append(c_idx)
        car_props.append({
            'path_idx': 0,
            'arrival_time': 0, 
            'state': 'queued'
        })
        
    # Init Lights
    int_states = {}
    for i_id, sched in intersection_schedules.items():
        if sched:
            int_states[i_id] = {'idx': 0, 't_rem': sched[0][1]}
            
    finished_count = 0
    current_score = 0
    
    for t in range(D + 1):
        
        # -- Process Intersections --
        for i_id, sched in intersection_schedules.items():
            if not sched: continue
            
            state = int_states[i_id]
            green_street_name, duration = sched[state['idx']]
            
            # Try to move ONE car
            street_q = queues[green_street_name]
            
            if street_q:
                c_idx = street_q[0]
                car = car_props[c_idx]
                
                # Check if physically arrived
                if car['arrival_time'] <= t:
                    street_q.popleft()
                    
                    # Advance Car
                    p_idx = car['path_idx'] + 1
                    path = car_paths[c_idx]
                    
                    if p_idx >= len(path):
                        # Finished
                        finished_count += 1
                        points = F + (D - t)
                        current_score += points
                        car['state'] = 'done'
                    else:
                        # Move to next street
                        next_s = path[p_idx]
                        travel = street_map[next_s][1]
                        car['path_idx'] = p_idx
                        car['arrival_time'] = t + travel
                        queues[next_s].append(c_idx)
            
            # -- Update Light Timer --
            state['t_rem'] -= 1
            if state['t_rem'] == 0:
                # Switch light
                state['idx'] = (state['idx'] + 1) % len(sched)
                state['t_rem'] = sched[state['idx']][1]

        # -- Logging --
        if t % PRINT_INTERVAL == 0 or t == D:
            print(f"{t:<10} | {finished_count:<15} | {current_score:<10}")

    print("-" * 50)
    print(f"Total Score: {current_score}")
    
    # ==========================================
    # 5. WRITE OUTPUT
    # ==========================================
    with open(OUTPUT_FILENAME, 'w') as f:
        f.write(f"{len(intersection_schedules)}\n")
        for i_id, sched in intersection_schedules.items():
            f.write(f"{i_id}\n")
            f.write(f"{len(sched)}\n")
            for name, duration in sched:
                f.write(f"{name} {duration}\n")
                
    print(f"File saved: {OUTPUT_FILENAME}")

if __name__ == "__main__":
    solve_time_ordered_greedy()